In [1]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plot
from sklearn.linear_model import LinearRegression
import sklearn.metrics
from sklearn.metrics import explained_variance_score, mean_squared_error, r2_score # https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html
import statsmodels.api as sm
import pandas
import numpy as np
import matplotlib.pyplot as plt
import pandas

# For 3d plots. This import is necessary to have 3D plotting below
from mpl_toolkits.mplot3d import Axes3D

# For statistics. Requires statsmodels 5.0 or more
from statsmodels.formula.api import ols
# Analysis of Variance (ANOVA) on linear models
from statsmodels.stats.anova import anova_lm

# Adapted from:
# https://www.datasciencecentral.com/profiles/blogs/linear-regression-in-python-use-of-numpy-scipy-and-statsmodels,
# https://realpython.com/linear-regression-in-python/#multiple-linear-regression


In [2]:
DataToFit = np.transpose(np.asarray(np.genfromtxt('TrabucchiPdataforHenryFit.csv',delimiter=',',skip_header=1)))
# 6 = logM
# 7 = logL
# 9 = logR
# 15 = Fundamental period from COLIBRI model
# 16 = logP0 from COLIBRI model
# 17 = logP0 from Trabucchi formula
# 19 = logP0 from Ostlie-Cox formula
# 22 = Overtone period from COLIBRI model
# 23 = logP1 from COLIBRI model
# 24 = logP1 from Trabucchi formula
# 25 = logP1 from Ostlie-Cox formula
# 26 = Pulsation Mode
# 35 = log(C/O) 

In [3]:
DataToFit[35]

array([-0.5717   , -0.57112  , -0.57099  , -0.57089  , -0.57084  ,
       -0.57047  , -0.57038  , -0.57023  , -0.57018  , -0.57012  ,
       -0.57     , -0.56644  , -0.56621  , -0.56317  , -0.56234  ,
       -0.56222  , -0.55793  , -0.55793  , -0.55637  , -0.55561  ,
       -0.55545  , -0.5554   , -0.55478  , -0.55433  , -0.55422  ,
       -0.55314  , -0.55185  , -0.55078  , -0.54981  , -0.5497   ,
       -0.54966  , -0.54847  , -0.54756  , -0.54531  , -0.54531  ,
       -0.54531  , -0.54531  , -0.54531  , -0.54343  , -0.5414   ,
       -0.5414   , -0.5414   , -0.5414   , -0.5414   , -0.53975  ,
       -0.53805  , -0.53805  , -0.53784  , -0.53495  , -0.53495  ,
       -0.53345  , -0.53209  , -0.53209  , -0.53081  , -0.53049  ,
       -0.52861  , -0.52861  , -0.52506  , -0.52506  , -0.5234   ,
       -0.52298  , -0.52236  , -0.52123  , -0.52123  , -0.5202   ,
       -0.51896  , -0.51886  , -0.51764  , -0.51753  , -0.51753  ,
       -0.5163   , -0.51546  , -0.51386  , -0.51386  , -0.5138

In [4]:
def PulsationModeSort (input_array):
    FundamentalPulsators = []
    OvertonePulsators = []
    i = 0
    while i < len(input_array[18]):
        if input_array[26][i] == 0:
            FundamentalPulsators.append(i)
        if input_array[26][i] == 1:
            OvertonePulsators.append(i)
        i += 1
    FundamentalModeStars = np.transpose(input_array.transpose()[FundamentalPulsators])
    FirstOvertoneModeStars = np.transpose(input_array.transpose()[OvertonePulsators])
    return FundamentalModeStars, FirstOvertoneModeStars

def CompositionSort (input_array):
    ORichStarsIndices = []
    CRichStarsIndices = []
    i = 0
    while i < len(input_array[35]):
        if input_array[35][i] < 0:
            ORichStarsIndices.append(i)
        if input_array[35][i] > 0:
            CRichStarsIndices.append(i)
        i += 1
    ORichStars = np.transpose(input_array.transpose()[ORichStarsIndices])
    CRichStars = np.transpose(input_array.transpose()[CRichStarsIndices])
    return ORichStars, CRichStars

In [5]:
P0Stars, P1Stars = PulsationModeSort(DataToFit)

ORichP0Stars, CRichP0Stars = CompositionSort(P0Stars)
ORichP1Stars, CRichP1Stars = CompositionSort(P1Stars)

In [6]:
# First, let's look at all of the Fundamental mode data at once.
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = P0Stars[6]
logRData = P0Stars[9]
logPData = P0Stars[17]

data = pandas.DataFrame({'x0': logMData, 'x1': logRData, 'y': logPData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                 1.889e+05
Date:                Sun, 31 Oct 2021   Prob (F-statistic):          7.85e-320
Time:                        22:02:12   Log-Likelihood:                 788.54
No. Observations:                 197   AIC:                            -1571.
Df Residuals:                     194   BIC:                            -1561.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.3747      0.008   -302.484      0.0

O & C stars: logP0 = -2.414(13) - 0.881(9)\*logM + 2.071(6)\*logR

In [7]:
# Now O-Rich
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = ORichP0Stars[6]
logRData = ORichP0Stars[9]
logPData = ORichP0Stars[17]

data = pandas.DataFrame({'x0': logMData, 'x1': logRData, 'y': logPData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.663e+05
Date:                Sun, 31 Oct 2021   Prob (F-statistic):          7.88e-177
Time:                        22:02:12   Log-Likelihood:                 484.17
No. Observations:                 103   AIC:                            -962.3
Df Residuals:                     100   BIC:                            -954.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.3136      0.008   -291.239      0.0

O stars: logP0 = -2.314(8) - 0.874(3)\*logM + 2.028(4)\*logR

In [8]:
# Now C-Rich
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = CRichP0Stars[6]
logRData = CRichP0Stars[9]
logPData = CRichP0Stars[17]

data = pandas.DataFrame({'x0': logMData, 'x1': logRData, 'y': logPData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.251e+05
Date:                Sun, 31 Oct 2021   Prob (F-statistic):          3.21e-157
Time:                        22:02:12   Log-Likelihood:                 402.54
No. Observations:                  94   AIC:                            -799.1
Df Residuals:                      91   BIC:                            -791.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.4861      0.010   -247.058      0.0

C stars: logP0 = -2.486(10) - 0.882(3)\*logM + 2.101(4)\*logR

In [9]:
# Now, let's look at all of the first overtone mode data at once.
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = P1Stars[6]
logRData = P1Stars[9]
logPData = P1Stars[17]

data = pandas.DataFrame({'x0': logMData, 'x1': logRData, 'y': logPData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.998
Model:                            OLS   Adj. R-squared:                  0.998
Method:                 Least Squares   F-statistic:                 4.761e+04
Date:                Sun, 31 Oct 2021   Prob (F-statistic):          3.34e-248
Time:                        22:02:12   Log-Likelihood:                 726.84
No. Observations:                 185   AIC:                            -1448.
Df Residuals:                     182   BIC:                            -1438.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.0652      0.013   -154.758      0.0

O & C stars: logP1 = -2.065(13) - 0.763(5)\*logM + 1.912(6)\*logR

In [10]:
# Now O-Rich
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = ORichP1Stars[6]
logRData = ORichP1Stars[9]
logPData = ORichP1Stars[17]

data = pandas.DataFrame({'x0': logMData, 'x1': logRData, 'y': logPData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                 1.016e+05
Date:                Sun, 31 Oct 2021   Prob (F-statistic):          8.45e-225
Time:                        22:02:12   Log-Likelihood:                 656.49
No. Observations:                 145   AIC:                            -1307.
Df Residuals:                     142   BIC:                            -1298.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.0609      0.009   -228.528      0.0

O stars: logP1 = -2.061(9) - 0.750(3)\*logM + 1.911(4)\*logR

In [11]:
# Now C-Rich
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = CRichP1Stars[6]
logRData = CRichP1Stars[9]
logPData = CRichP1Stars[17]

data = pandas.DataFrame({'x0': logMData, 'x1': logRData, 'y': logPData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                 2.982e+04
Date:                Sun, 31 Oct 2021   Prob (F-statistic):           4.56e-60
Time:                        22:02:12   Log-Likelihood:                 172.50
No. Observations:                  40   AIC:                            -339.0
Df Residuals:                      37   BIC:                            -333.9
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.1622      0.017   -125.676      0.0

C stars: logP1 = -2.162(17) - 0.748(12)\*logM + 1.952(8)\*logR